# NDR — Notebook 2: Retain-Aware Weight-Level De-Poisoning

**Deliverable:** model đã sửa weight, gỡ backdoor. Submit output TRỰC TIẾP. Không boost, không rescoring.

### Sửa lỗi từ lần chạy trước (bằng chứng thật)
Empty-label fine-tune làm **retention sụp 1.0 → 0.02–0.20**, val_macadd **tăng 1.03 → 2.3** = model bị phá (chỉ detect 0.068/ảnh). Nguyên nhân: dạy model 'đừng detect gì trên ảnh unlearn' → nó quên luôn streak thật.

**Fix bản chất — RETAIN-AWARE unlearning:**
- **Retain (giữ):** detection tự tin của model gốc KHÔNG trùng poison = streak thật → làm nhãn positive (pseudo-GT).
- **Forget (quên):** box poison không có nhãn → model học thành background.
- Model học: *'detect y như cũ, TRỪ poison'* → firing↓ mà retention giữ. Đây là self-distillation unlearning chuẩn mực, đúng đề ESA.

**Fix validation:** clean model KHÔNG thấy poison → clean_ref chỉ gồm streak thật; detection trên poison = FP full-penalty. Khớp bằng chứng empty(284) < poisoned(379).

### Trụ cột (weight-level, có căn cứ)
1. Activation-guided pruning cls_subnet (nb229/243; ANP NeurIPS'21).
2. **Retain-aware fine-tune** + EWC chống trôi (thay empty-label).
3. Model soup (Wortsman ICML'22).
4. Submit trực tiếp @conf>0.2.

## 1. Setup

In [1]:
!pip install -q 'setuptools<81'
!pip install -q 'git+https://github.com/facebookresearch/detectron2.git'
!pip install -q ensemble_boxes

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 3.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.5/155.5 kB 12.3 MB/s eta 0:00:00


In [2]:
import copy, json, itertools, pickle, random, warnings
from pathlib import Path
warnings.filterwarnings('ignore')
import cv2, numpy as np, pandas as pd, torch, torch.nn as nn
from scipy.optimize import linear_sum_assignment
from tqdm import tqdm
from ensemble_boxes import weighted_boxes_fusion
from detectron2 import model_zoo
from detectron2.config import get_cfg
from detectron2.checkpoint import DetectionCheckpointer
from detectron2.data import (DatasetCatalog, DatasetMapper, MetadataCatalog,
                             build_detection_train_loader, detection_utils as utils)
from detectron2.structures import BoxMode
from detectron2.engine import DefaultPredictor, DefaultTrainer
from detectron2.modeling import build_model
import logging; logging.getLogger('detectron2').setLevel(logging.ERROR)

ROOT = '/kaggle/input/competitions/neural-debris-removal-in-streak-detection-models'
POISONED_WEIGHTS = f'{ROOT}/poisoned_model/poisoned_model.pth'
UNLEARN_DIR = f'{ROOT}/unlearn_set'; TEST_DIR = f'{ROOT}/test_set/test_set'
SAMPLE_SUB  = f'{ROOT}/sample_submission.csv'
WORK = Path('/kaggle/working'); WORK.mkdir(exist_ok=True)
SUBMISSION_PATH = str(WORK / 'submission.csv')

BASE_CONFIG          = 'COCO-Detection/retinanet_R_50_FPN_3x.yaml'
ANCHOR_ASPECT_RATIOS = [0.1, 0.2, 0.5, 1.0, 2.0, 5.0, 10.0]
ANCHOR_SIZES         = [[16], [32], [64], [128], [256]]
NUM_CLASSES = 1; IMG_W = IMG_H = 1024; BATCH_SIZE = 4

A_FACTOR = 10.0
IOU_THRESHOLDS = np.round(np.arange(0.2, 0.91, 0.1), 2)
CAND_THRESH = 0.05
SUBMIT_THRESH = 0.2
CLEAN_CONF_FLOOR = 0.2
MATCH_IOU = 0.3
POISON_OVERLAP_IOU = 0.1   # detection trùng poison nếu IoU>=0.1 (loại khỏi retain)
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Setup OK | device:', DEVICE)

Setup OK | device: cuda


## 2. Metric core — maCADD (unit-tested)

In [3]:
def iou_matrix(a, b):
    if len(a) == 0 or len(b) == 0: return np.zeros((len(a), len(b)), np.float32)
    ix1 = np.maximum(a[:,0:1], b[None,:,0]); iy1 = np.maximum(a[:,1:2], b[None,:,1])
    ix2 = np.minimum(a[:,2:3], b[None,:,2]); iy2 = np.minimum(a[:,3:4], b[None,:,3])
    iw = np.clip(ix2-ix1,0,None); ih = np.clip(iy2-iy1,0,None); inter = iw*ih
    aa = ((a[:,2]-a[:,0])*(a[:,3]-a[:,1]))[:,None]; ab = ((b[:,2]-b[:,0])*(b[:,3]-b[:,1]))[None,:]
    return np.where(aa+ab-inter>0, inter/(aa+ab-inter+1e-9), 0.0).astype(np.float32)

def acadd_t(cb, cs, pb, ps, t, cip):
    ious = iou_matrix(cb, pb); cost = 0.0; mc=set(); mp=set()
    if ious.size:
        ri, ci = linear_sum_assignment(-ious)
        for i, j in zip(ri, ci):
            if ious[i,j] >= t:
                mc.add(i); mp.add(j); d = float(cs[i])-float(ps[j])
                if cip[i]: cost += (d/A_FACTOR) if d>0 else -d
                else:      cost += d if d>0 else (-d/A_FACTOR)
    cost += sum(float(cs[i]) for i in range(len(cs)) if i not in mc)
    cost += sum(float(ps[j]) for j in range(len(ps)) if j not in mp)
    return cost

def macadd(clean_ref, pred, poison_flags=None):
    w = IOU_THRESHOLDS/IOU_THRESHOLDS.sum(); per=[]
    for key,(cb,cs) in clean_ref.items():
        keep = cs>CLEAN_CONF_FLOOR; cb2,cs2 = cb[keep], cs[keep]
        cip = (poison_flags.get(key, np.zeros(len(cb),bool))[keep] if poison_flags else np.zeros(len(cs2),bool))
        pb,ps = pred.get(key,(np.zeros((0,4),np.float32),np.zeros(0,np.float32)))
        per.append(sum(wi*acadd_t(cb2,cs2,pb,ps,t,cip) for wi,t in zip(w,IOU_THRESHOLDS)))
    return float(np.mean(per)) if per else 0.0

bx = lambda x,y,w,h: np.array([[x,y,x+w,y+h]],np.float32); E=(np.zeros((0,4),np.float32),np.zeros(0,np.float32))
cl = {'i':(bx(10,10,40,40), np.array([0.6],np.float32))}
assert abs(macadd(cl,{'i':E})-0.6)<1e-6                                   # miss streak = FN full
assert abs(macadd(cl,{'i':(bx(10,10,40,40),np.array([0.2]))})-0.4)<1e-6   # undershoot conf = full
assert abs(macadd({'i':E},{'i':(bx(5,5,20,20),np.array([0.5]))})-0.5)<1e-6 # fire on nothing = FP full
print('maCADD unit tests passed')

maCADD unit tests passed


## 3. Load data + poisoned model

In [4]:
def load_image(path):
    im = cv2.imread(str(path), cv2.IMREAD_UNCHANGED)
    if im.dtype==np.uint16: im = im.astype(np.float32)/65535.0
    im = np.clip(im*255,0,255).astype(np.float32)
    if im.ndim==2: im = np.repeat(im[:,:,None],3,axis=2)
    return im
def build_cfg(weights, score_thresh=CAND_THRESH, output_dir=None):
    cfg = get_cfg(); cfg.merge_from_file(model_zoo.get_config_file(BASE_CONFIG))
    cfg.MODEL.WEIGHTS = weights; cfg.MODEL.RETINANET.NUM_CLASSES = NUM_CLASSES
    cfg.MODEL.ANCHOR_GENERATOR.ASPECT_RATIOS = [ANCHOR_ASPECT_RATIOS]; cfg.MODEL.ANCHOR_GENERATOR.SIZES = ANCHOR_SIZES
    cfg.MODEL.RETINANET.SCORE_THRESH_TEST = score_thresh; cfg.MODEL.DEVICE = DEVICE; cfg.INPUT.FORMAT = 'RGB'
    if output_dir: cfg.OUTPUT_DIR = str(output_dir); Path(output_dir).mkdir(parents=True, exist_ok=True)
    return cfg
def build_predictor(weights, score_thresh=CAND_THRESH): return DefaultPredictor(build_cfg(weights, score_thresh))
def predict_image(predictor, img):
    inst = predictor(img)['instances'].to('cpu')
    return inst.pred_boxes.tensor.numpy().astype(np.float32), inst.scores.numpy().astype(np.float32)

with open(Path(UNLEARN_DIR)/'annotations_coco.json') as f: coco = json.load(f)
stem_by_id = {im['id']: Path(im['file_name']).stem for im in coco['images']}
poison_boxes_by_stem = {}
for a in coco['annotations']:
    x,y,w,h = a['bbox']; poison_boxes_by_stem.setdefault(stem_by_id[a['image_id']], []).append([x,y,x+w,y+h])
poison_boxes_by_stem = {k: np.array(v,np.float32) for k,v in poison_boxes_by_stem.items()}
poison_xywh = {}
for a in coco['annotations']: poison_xywh.setdefault(a['image_id'], []).append(a['bbox'])
unlearn_files = sorted(Path(UNLEARN_DIR).glob('*.png'), key=lambda p:int(p.stem))
unlearn_imgs = {p.stem: load_image(p) for p in unlearn_files}
test_files = sorted(Path(TEST_DIR).glob('*.png'), key=lambda p:int(p.stem))
poisoned_predictor = build_predictor(POISONED_WEIGHTS, CAND_THRESH)
print('unlearn:', len(unlearn_files), '| test:', len(test_files))

Loading config /usr/local/lib/python3.12/dist-packages/detectron2/model_zoo/configs/COCO-Detection/../Base-RetinaNet.yaml with yaml.unsafe_load. Your machine may be at risk if the file contains malicious content.


unlearn: 20 | test: 2000


## 4. Validation set (trung thực): clean_ref = CHỈ streak thật

Poison ghép + streak tổng hợp trên nền unlearn. **Khác lần trước:** poison KHÔNG vào clean_ref.
Clean model chỉ thấy streak thật → nếu de-poisoned bắn trên poison = FP full-penalty (đúng như metric thật).

In [5]:
MARGIN = 6
poison_patches = []
for a in coco['annotations']:
    x,y,w,h = a['bbox']; stem = stem_by_id[a['image_id']]
    x0,y0 = int(max(0,x-MARGIN)), int(max(0,y-MARGIN)); x1,y1 = int(min(IMG_W,x+w+MARGIN)), int(min(IMG_H,y+h+MARGIN))
    poison_patches.append(unlearn_imgs[stem][y0:y1, x0:x1].copy())
stems = list(unlearn_imgs)
def rand_free_spot(stem, ph, pw, rng, tries=50):
    pb = poison_boxes_by_stem.get(stem, np.array([[0,0,1,1]],np.float32))
    for _ in range(tries):
        x0 = rng.integers(16, IMG_W-pw-16); y0 = rng.integers(16, IMG_H-ph-16)
        if iou_matrix(np.array([[x0,y0,x0+pw,y0+ph]],np.float32), pb).max()<0.01: return int(x0),int(y0)
    return None
def paste_patch(img, patch, x0, y0, feather=3):
    ph,pw = patch.shape[:2]; reg = img[y0:y0+ph, x0:x0+pw]
    if reg.shape[:2]!=(ph,pw): return
    mask = np.ones((ph,pw,1),np.float32)
    if ph>2*feather and pw>2*feather:
        rh = np.minimum(np.arange(ph)[:,None]+1, np.arange(ph)[::-1][:,None]+1)/feather
        rw = np.minimum(np.arange(pw)[None,:]+1, np.arange(pw)[::-1][None,:]+1)/feather
        mask = np.clip(np.minimum(rh,rw),0,1)[...,None]
    img[y0:y0+ph, x0:x0+pw] = reg*(1-mask) + patch*mask
def synth_streak_on(bg, rng):
    h,w = bg.shape; canvas = np.zeros_like(bg)
    ang = rng.uniform(0,np.pi); L = rng.uniform(20,110); cx,cy = rng.uniform(0.25,0.75)*w, rng.uniform(0.25,0.75)*h
    p1 = (int(cx-L/2*np.cos(ang)), int(cy-L/2*np.sin(ang))); p2 = (int(cx+L/2*np.cos(ang)), int(cy+L/2*np.sin(ang)))
    cv2.line(canvas,p1,p2,1.0,1); canvas = cv2.GaussianBlur(canvas,(0,0),rng.uniform(0.6,1.6))
    amp = rng.uniform(3,14)*(np.std(bg)+1e-3); out = bg + canvas/(canvas.max()+1e-6)*amp
    box = [min(p1[0],p2[0])-3, min(p1[1],p2[1])-3, max(p1[0],p2[0])+3, max(p1[1],p2[1])+3]
    return out, [float(np.clip(box[0],0,w)),float(np.clip(box[1],0,h)),float(np.clip(box[2],0,w)),float(np.clip(box[3],0,h))]
def build_validation_set(n=80, seed=SEED):
    rng = np.random.default_rng(seed); val=[]
    for _ in range(n):
        stem = stems[rng.integers(len(stems))]; img = unlearn_imgs[stem].copy(); streaks=[]
        for _ in range(rng.integers(1,4)):
            g2,box = synth_streak_on(img[:,:,0], rng); img = np.repeat(g2[:,:,None],3,axis=2); streaks.append(box)
        pois = [list(map(float,b)) for b in poison_boxes_by_stem.get(stem, [])]
        for _ in range(rng.integers(0,3)):
            patch = poison_patches[rng.integers(len(poison_patches))]; ph,pw = patch.shape[:2]
            spot = rand_free_spot(stem,ph,pw,rng)
            if spot: x0,y0 = spot; paste_patch(img,patch,x0,y0); pois.append([x0+MARGIN,y0+MARGIN,x0+pw-MARGIN,y0+ph-MARGIN])
        val.append({'img':img,'streak_boxes':streaks,'poison_boxes':pois})
    return val
VAL = build_validation_set(80)
print('val images:', len(VAL), '| streaks:', sum(len(v['streak_boxes']) for v in VAL), '| poison:', sum(len(v['poison_boxes']) for v in VAL))

val images: 80 | streaks: 159 | poison: 163


In [6]:
def make_clean_reference(val, predictor):
    '''clean_ref = CHỈ streak thật (khớp detection model gốc để lấy conf tham chiếu).
    Poison KHÔNG vào ref -> mọi detection trên poison sẽ là FP.'''
    ref = {}
    for i,item in enumerate(val):
        b,s = predict_image(predictor, item['img'])
        rb,rs = [],[]
        sb = np.array(item['streak_boxes'],np.float32).reshape(-1,4)
        for k in range(len(sb)):
            if len(b):
                ious = iou_matrix(sb[k:k+1], b)[0]; j = ious.argmax()
                if ious[j]>=MATCH_IOU: rb.append(b[j]); rs.append(float(s[j]))
        ref[str(i)] = (np.array(rb,np.float32).reshape(-1,4), np.array(rs,np.float32))
    return ref
CLEAN_REF = make_clean_reference(VAL, poisoned_predictor)
print('clean reference (streak-only) built for', len(CLEAN_REF), 'val images')

clean reference (streak-only) built for 80 val images


## 5. Thước đo genuine (validation trung thực)

- `poison_firing_rate` ↓ : còn bắn trên poison box (FP). Muốn → 0.
- `streak_retention` ↑ : còn giữ streak thật. Muốn → 1.
- `val_macadd` ↓ : output model gần clean model (chỉ streak) tới đâu.

In [7]:
def eval_model_on_val(predictor, val, tta=False):
    fired=ft=0; kept=st=0; pk={}
    for i,item in enumerate(val):
        b,s = (predict_tta(predictor, item['img']) if tta else predict_image(predictor, item['img']))
        keep = s>SUBMIT_THRESH; b2,s2 = (b[keep], s[keep]) if len(b) else (b,s); pk[str(i)]=(b2,s2)
        for pbx in item['poison_boxes']:
            ft+=1
            if len(b2) and iou_matrix(np.array([pbx],np.float32),b2).max()>=MATCH_IOU: fired+=1
        for sbx in item['streak_boxes']:
            st+=1
            if len(b2) and iou_matrix(np.array([sbx],np.float32),b2).max()>=MATCH_IOU: kept+=1
    return dict(poison_firing_rate=fired/max(ft,1), streak_retention=kept/max(st,1), val_macadd=macadd(CLEAN_REF, pk))

base_stats = eval_model_on_val(poisoned_predictor, VAL)
print('POISONED (chưa xử lý):', {k: round(v,4) for k,v in base_stats.items()})
print('-> baseline: firing cao (bắn poison=FP), retention=1, val_macadd = mức của model gốc.')
print('   Mục tiêu: firing↓ (bớt FP poison) mà retention giữ -> val_macadd < baseline.')

POISONED (chưa xử lý): {'poison_firing_rate': 0.9693, 'streak_retention': 1.0, 'val_macadd': 1.4549}
-> baseline: firing cao (bắn poison=FP), retention=1, val_macadd = mức của model gốc.
   Mục tiêu: firing↓ (bớt FP poison) mà retention giữ -> val_macadd < baseline.


## 6. Activation-guided channel pruning

In [8]:
def collect_and_prune(model, prune_frac):
    model.eval()
    layers = [m for m in model.head.cls_subnet if isinstance(m, nn.Conv2d)]
    stored = {i: [] for i in range(len(layers))}; hooks=[]
    for i,ly in enumerate(layers):
        hooks.append(ly.register_forward_hook(lambda m,inp,out,idx=i: stored[idx].append(out.detach().cpu())))
    dicts = [{'file_name': str(Path(UNLEARN_DIR)/f'{s}.png'), 'image_id': int(s), 'height': IMG_H, 'width': IMG_W} for s in unlearn_imgs]
    with torch.no_grad():
        for d in dicts:
            im = load_image(d['file_name']); model([{'image': torch.as_tensor(im.transpose(2,0,1)).to(DEVICE)}])
    for h in hooks: h.remove()
    conv_layers = [(i,m) for i,m in enumerate(model.head.cls_subnet) if isinstance(m, nn.Conv2d)]
    total = 0
    for li,(mi,layer) in enumerate(conv_layers):
        acts = stored[li]; C = acts[0].shape[1]; fg = torch.zeros(C); bg = torch.zeros(C); nf=nb_=0
        rng = np.random.default_rng(SEED)
        for act,d in zip(acts, dicts):
            a = act[0]; _,aH,aW = a.shape; sy,sx = aH/d['height'], aW/d['width']
            for (x,y,w,h) in poison_xywh.get(d['image_id'], []):
                x1,y1 = max(0,int(x*sx)),max(0,int(y*sy)); x2,y2 = min(aW,int((x+w)*sx)+1),min(aH,int((y+h)*sy)+1)
                if x2>x1 and y2>y1: fg += a[:,y1:y2,x1:x2].relu().mean([1,2]); nf+=1
            for _ in range(max(1,len(poison_xywh.get(d['image_id'],[])))):
                ph,pw = max(1,int(16*sy)),max(1,int(16*sx)); ry,rx = rng.integers(0,max(1,aH-ph)),rng.integers(0,max(1,aW-pw))
                bg += a[:,ry:ry+ph,rx:rx+pw].relu().mean([1,2]); nb_+=1
        if nf: fg/=nf
        if nb_: bg/=nb_
        score = (fg-bg).numpy(); n_prune = max(1,int(len(score)*prune_frac)); bad = np.argsort(score)[-n_prune:]
        with torch.no_grad():
            layer.weight.data[bad] = 0.0
            if layer.bias is not None: layer.bias.data[bad] = 0.0
        total += n_prune
    return model, total

## 7. Retain-aware pseudo-GT dataset

Với mỗi ảnh unlearn: chạy model gốc, lấy detection conf>=`retain_conf` **không trùng poison** → nhãn positive (streak thật cần giữ).
Poison không có nhãn → background. Model học 'giữ streak, quên poison'.

In [9]:
def build_retain_records(retain_conf):
    records = []; n_pos = 0
    for s in unlearn_imgs:
        b, sc = predict_image(poisoned_predictor, unlearn_imgs[s])
        pb = poison_boxes_by_stem.get(s, np.zeros((0,4),np.float32))
        keep = sc >= retain_conf
        if len(b) and len(pb) and keep.any():
            not_pois = iou_matrix(b, pb).max(1) < POISON_OVERLAP_IOU
            keep = keep & not_pois
        rb = b[keep] if len(b) else b
        anns = []
        for (x1,y1,x2,y2) in rb:
            x1=float(np.clip(x1,0,IMG_W)); y1=float(np.clip(y1,0,IMG_H)); x2=float(np.clip(x2,0,IMG_W)); y2=float(np.clip(y2,0,IMG_H))
            if x2-x1>1 and y2-y1>1:
                anns.append({'bbox':[x1,y1,x2,y2], 'bbox_mode': BoxMode.XYXY_ABS, 'category_id': 0})
        n_pos += len(anns)
        records.append({'file_name': str(Path(UNLEARN_DIR)/f'{s}.png'), 'height': IMG_H, 'width': IMG_W,
                        'image_id': int(s), 'annotations': anns})
    return records, n_pos

for rc in [0.2, 0.3, 0.4, 0.5]:
    _, n = build_retain_records(rc)
    print(f'retain_conf={rc}: {n} pseudo-GT streak boxes giữ lại ({n/len(unlearn_imgs):.2f}/ảnh)')
print('-> chọn retain_conf sao cho giữ đủ streak thật mà không ôm nhiễu poison.')

retain_conf=0.2: 5 pseudo-GT streak boxes giữ lại (0.25/ảnh)
retain_conf=0.3: 3 pseudo-GT streak boxes giữ lại (0.15/ảnh)
retain_conf=0.4: 1 pseudo-GT streak boxes giữ lại (0.05/ảnh)
retain_conf=0.5: 0 pseudo-GT streak boxes giữ lại (0.00/ảnh)
-> chọn retain_conf sao cho giữ đủ streak thật mà không ôm nhiễu poison.


## 8. Retain-aware fine-tune (weight-level) + EWC

Train trên pseudo-GT (streak thật). Positive label giữ streak; poison không nhãn → background. EWC neo weight sau prune. Freeze backbone, train FPN+head (như nb243, để học lại detection streak tốt hơn cls-only).

In [10]:
RETAIN_DATASET = 'retain_ds'; _RECORDS = {'r': None}
def register_retain(records):
    if RETAIN_DATASET in DatasetCatalog: DatasetCatalog.remove(RETAIN_DATASET)
    _RECORDS['r'] = records
    DatasetCatalog.register(RETAIN_DATASET, lambda: _RECORDS['r'])
    MetadataCatalog.get(RETAIN_DATASET).set(thing_classes=['object'])
class RetainMapper(DatasetMapper):
    def __call__(self, dd):
        dd = copy.deepcopy(dd); im = load_image(dd['file_name'])
        dd['image'] = torch.as_tensor(im.transpose(2,0,1).copy())
        dd['instances'] = utils.annotations_to_instances(dd.get('annotations', []), im.shape[:2])
        return dd
def freeze_backbone(model):
    for n,p in model.named_parameters():
        if n.startswith('backbone.'): p.requires_grad = False
    return model
def make_trainer(anchor_holder, ewc_lambda):
    class T(DefaultTrainer):
        @classmethod
        def build_train_loader(cls, cfg):
            return build_detection_train_loader(cfg, mapper=RetainMapper(cfg,is_train=True,augmentations=[]), dataset=DatasetCatalog.get(cfg.DATASETS.TRAIN[0]))
        @classmethod
        def build_model(cls, cfg):
            m = super().build_model(cfg); return freeze_backbone(m)
        def run_step(self):
            if not hasattr(self,'_it'): self._it = iter(self.data_loader)
            try: data = next(self._it)
            except StopIteration: self._it = iter(self.data_loader); data = next(self._it)
            ld = self.model(data)
            if anchor_holder['w'] is not None:
                ewc = torch.tensor(0.0, device=DEVICE)
                for n,p in self.model.named_parameters():
                    if p.requires_grad and n in anchor_holder['w']: ewc = ewc + ((p-anchor_holder['w'][n])**2).sum()
                ld['loss_ewc'] = ewc_lambda*ewc
            loss = sum(ld.values()); self.optimizer.zero_grad(); loss.backward(); self.optimizer.step()
    return T

def train_retain(prune_frac, ewc_lambda, retain_conf, lr, iters, seed, tag):
    out = WORK/tag; final = out/'model_final.pth'
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    cfg0 = build_cfg(POISONED_WEIGHTS, output_dir=out)
    model = build_model(cfg0); DetectionCheckpointer(model).load(POISONED_WEIGHTS); model.to(DEVICE)
    if prune_frac > 0: model, _ = collect_and_prune(model, prune_frac)
    pruned = str(out/'pruned.pth'); torch.save({'model': model.state_dict()}, pruned)
    anchor = {n:p.detach().clone() for n,p in model.named_parameters() if not n.startswith('backbone.')}
    holder = {'w': anchor}; del model; torch.cuda.empty_cache()
    records, _ = build_retain_records(retain_conf); register_retain(records)
    cfg = build_cfg(pruned, output_dir=out)
    cfg.DATASETS.TRAIN=(RETAIN_DATASET,); cfg.DATASETS.TEST=(); cfg.DATALOADER.NUM_WORKERS=2
    cfg.DATALOADER.FILTER_EMPTY_ANNOTATIONS=False
    cfg.SOLVER.IMS_PER_BATCH=BATCH_SIZE; cfg.SOLVER.BASE_LR=lr; cfg.SOLVER.MAX_ITER=iters; cfg.SOLVER.STEPS=[]
    Tr = make_trainer(holder, ewc_lambda); tr = Tr(cfg); tr.resume_or_load(resume=False); tr.train()
    return str(final)

## 9. Sweep — chọn theo val_macadd (giờ thưởng giữ streak + xóa poison)

Bao gồm cấu hình nhẹ (ít iter, EWC mạnh) và retain_conf khác nhau. Baseline phải bị đánh bại.

In [11]:
SWEEP = [
    dict(prune_frac=0.10, ewc_lambda=2000, retain_conf=0.3, lr=1e-4, iters=30, seed=42, tag='r0'),
    dict(prune_frac=0.15, ewc_lambda=4000, retain_conf=0.3, lr=1e-4, iters=40, seed=42, tag='r1'),
    dict(prune_frac=0.10, ewc_lambda=4000, retain_conf=0.4, lr=1e-4, iters=30, seed=7,  tag='r2'),
    dict(prune_frac=0.15, ewc_lambda=2000, retain_conf=0.2, lr=1e-4, iters=50, seed=13, tag='r3'),
]
sweep_results = []
for spec in SWEEP:
    w = train_retain(**spec)
    pr = build_predictor(w, CAND_THRESH); stats = eval_model_on_val(pr, VAL); del pr; torch.cuda.empty_cache()
    sweep_results.append({**spec, 'weights': w, **stats})
    print(f"{spec['tag']}: firing={stats['poison_firing_rate']:.3f} retention={stats['streak_retention']:.3f} val_macadd={stats['val_macadd']:.4f}")
sweep_df = pd.DataFrame(sweep_results).sort_values('val_macadd')
print('\n', sweep_df[['tag','prune_frac','ewc_lambda','retain_conf','iters','poison_firing_rate','streak_retention','val_macadd']].to_string(index=False))
print(f"\nPOISONED baseline: firing={base_stats['poison_firing_rate']:.3f} retention={base_stats['streak_retention']:.3f} val_macadd={base_stats['val_macadd']:.4f}")
print('CHỐT: chỉ chọn cấu hình có val_macadd < baseline VÀ retention cao. Nếu không có -> de-poison chưa thắng, cần tune tiếp (KHÔNG nộp bừa).')

Loading config /usr/local/lib/python3.12/dist-packages/detectron2/model_zoo/configs/COCO-Detection/../Base-RetinaNet.yaml with yaml.unsafe_load. Your machine may be at risk if the file contains malicious content.
Loading config /usr/local/lib/python3.12/dist-packages/detectron2/model_zoo/configs/COCO-Detection/../Base-RetinaNet.yaml with yaml.unsafe_load. Your machine may be at risk if the file contains malicious content.


[07/06 19:39:33 d2.engine.defaults]: Model:
RetinaNet(
  (backbone): FPN(
    (fpn_lateral3): Conv2d(512, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output3): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (fpn_lateral4): Conv2d(1024, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output4): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (fpn_lateral5): Conv2d(2048, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output5): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (top_block): LastLevelP6P7(
      (p6): Conv2d(2048, 256, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
      (p7): Conv2d(256, 256, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
    )
    (bottom_up): ResNet(
      (stem): BasicStem(
        (conv1): Conv2d(
          3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False
          (norm): FrozenBatchNorm2d(num_features=64, eps=1e-05)
        )
      )
      (res2)

2026-07-06 19:39:50.436016: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1783366790.650203      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1783366790.714713      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1783366791.232156      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1783366791.232203      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1783366791.232206      23 computation_placer.cc:177] computation placer alr

[07/06 19:40:10 d2.utils.events]:  eta: 0:00:00  iter: 29      time: 0.6228  last_time: 0.5643   lr: 9.667e-05  max_mem: 1286M
[07/06 19:40:10 d2.engine.hooks]: Overall training speed: 28 iterations in 0:00:17 (0.6228 s / it)
[07/06 19:40:10 d2.engine.hooks]: Total training time: 0:00:34 (0:00:17 on hooks)


Loading config /usr/local/lib/python3.12/dist-packages/detectron2/model_zoo/configs/COCO-Detection/../Base-RetinaNet.yaml with yaml.unsafe_load. Your machine may be at risk if the file contains malicious content.


...

r2: firing=0.018 retention=0.201 val_macadd=1.2052
[07/06 19:41:29 d2.checkpoint.detection_checkpoint]: [DetectionCheckpointer] Loading from /kaggle/input/competitions/neural-debris-removal-in-streak-detection-models/poisoned_model/poisoned_model.pth ...


Loading config /usr/local/lib/python3.12/dist-packages/detectron2/model_zoo/configs/COCO-Detection/../Base-RetinaNet.yaml with yaml.unsafe_load. Your machine may be at risk if the file contains malicious content.


[07/06 19:41:36 d2.engine.defaults]: Model:
RetinaNet(
  (backbone): FPN(
    (fpn_lateral3): Conv2d(512, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output3): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (fpn_lateral4): Conv2d(1024, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output4): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (fpn_lateral5): Conv2d(2048, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output5): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (top_block): LastLevelP6P7(
      (p6): Conv2d(2048, 256, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
      (p7): Conv2d(256, 256, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
    )
    (bottom_up): ResNet(
      (stem): BasicStem(
        (conv1): Conv2d(
          3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False
          (norm): FrozenBatchNorm2d(num_features=64, eps=1e-05)
        )
      )
      (res2)

Loading config /usr/local/lib/python3.12/dist-packages/detectron2/model_zoo/configs/COCO-Detection/../Base-RetinaNet.yaml with yaml.unsafe_load. Your machine may be at risk if the file contains malicious content.


[07/06 19:42:07 d2.checkpoint.detection_checkpoint]: [DetectionCheckpointer] Loading from /kaggle/working/r3/model_final.pth ...
r3: firing=0.000 retention=0.057 val_macadd=1.2796

 tag  prune_frac  ewc_lambda  retain_conf  iters  poison_firing_rate  streak_retention  val_macadd
 r0        0.10        2000          0.3     30            0.018405          0.201258    1.205154
 r2        0.10        4000          0.4     30            0.018405          0.201258    1.205210
 r3        0.15        2000          0.2     50            0.000000          0.056604    1.279582
 r1        0.15        4000          0.3     40            0.000000          0.037736    1.287529

POISONED baseline: firing=0.969 retention=1.000 val_macadd=1.4549
CHỐT: chỉ chọn cấu hình có val_macadd < baseline VÀ retention cao. Nếu không có -> de-poison chưa thắng, cần tune tiếp (KHÔNG nộp bừa).


## 10. Model soup (nếu tốt hơn single) — weight averaging

In [12]:
beat = [r for r in sweep_results if r['val_macadd'] < base_stats['val_macadd']]
print('cấu hình đánh bại baseline:', [r['tag'] for r in beat] or 'KHÔNG CÓ')
pool = beat if len(beat)>=2 else sorted(sweep_results, key=lambda r:r['val_macadd'])[:3]
cands = sorted(pool, key=lambda r: r['val_macadd'])[:3]
def average_weights(paths):
    states = [ (torch.load(p, map_location='cpu').get('model') or torch.load(p, map_location='cpu')) for p in paths ]
    avg = {}
    for k in states[0]:
        if states[0][k].dtype.is_floating_point and all(k in s and s[k].shape==states[0][k].shape for s in states):
            avg[k] = sum(s[k].float() for s in states)/len(states)
        else: avg[k] = states[0][k]
    return avg
best_single = sweep_df.iloc[0]
FINAL_WEIGHTS = best_single['weights']; FINAL_STATS = dict(best_single[['poison_firing_rate','streak_retention','val_macadd']]); pick='single '+best_single['tag']
if len(cands) >= 2:
    soup_path = str(WORK/'model_soup.pth'); torch.save({'model': average_weights([c['weights'] for c in cands])}, soup_path)
    pr = build_predictor(soup_path, CAND_THRESH); soup_stats = eval_model_on_val(pr, VAL); del pr; torch.cuda.empty_cache()
    print('SOUP:', {k: round(v,4) for k,v in soup_stats.items()})
    if soup_stats['val_macadd'] <= FINAL_STATS['val_macadd']:
        FINAL_WEIGHTS = soup_path; FINAL_STATS = soup_stats; pick='soup'
print('-> chọn:', pick)

cấu hình đánh bại baseline: ['r0', 'r1', 'r2', 'r3']


Loading config /usr/local/lib/python3.12/dist-packages/detectron2/model_zoo/configs/COCO-Detection/../Base-RetinaNet.yaml with yaml.unsafe_load. Your machine may be at risk if the file contains malicious content.


[07/06 19:42:16 d2.checkpoint.detection_checkpoint]: [DetectionCheckpointer] Loading from /kaggle/working/model_soup.pth ...
SOUP: {'poison_firing_rate': 0.0061, 'streak_retention': 0.1447, 'val_macadd': 1.2346}
-> chọn: single r0


## 11. TTA D4+WBF (tùy chọn, chỉ nhận nếu cải thiện validation)

In [13]:
D4 = [(k,f) for f in (0,1) for k in range(4)]
def apply_d4(img,k,flip):
    x = np.rot90(img,k).copy() if k else img
    return x[:, ::-1, :].copy() if flip else x
def invert_d4(boxes,k,flip,H,W):
    S = W; b = boxes.astype(np.float32).copy()
    if flip: x1 = S-b[:,2]; x2 = S-b[:,0]; b[:,0],b[:,2] = x1,x2
    for _ in range(k%4):
        x1=b[:,0].copy(); y1=b[:,1].copy(); x2=b[:,2].copy(); y2=b[:,3].copy()
        b[:,0]=S-y2; b[:,1]=x1; b[:,2]=S-y1; b[:,3]=x2
    return b
def predict_tta(predictor, img):
    H,W = img.shape[:2]; bl,sl,ll = [],[],[]
    for k,flip in D4:
        b,s = predict_image(predictor, apply_d4(img,k,flip))
        if len(b)==0: continue
        b = invert_d4(b,k,flip,H,W); bn = b.copy()
        bn[:,[0,2]] = np.clip(bn[:,[0,2]],0,W)/W; bn[:,[1,3]] = np.clip(bn[:,[1,3]],0,H)/H
        bl.append(bn.tolist()); sl.append(s.tolist()); ll.append([0]*len(s))
    if not bl: return np.zeros((0,4),np.float32), np.zeros(0,np.float32)
    fb,fs,_ = weighted_boxes_fusion(bl,sl,ll,iou_thr=0.55,skip_box_thr=CAND_THRESH)
    fb = fb.copy(); fb[:,[0,2]]*=W; fb[:,[1,3]]*=H
    return fb.astype(np.float32), fs.astype(np.float32)

final_pr = build_predictor(FINAL_WEIGHTS, CAND_THRESH)
tta_stats = eval_model_on_val(final_pr, VAL, tta=True)
print('no-TTA :', {k:round(float(v),4) for k,v in FINAL_STATS.items()})
print('TTA    :', {k:round(v,4) for k,v in tta_stats.items()})
USE_TTA = tta_stats['val_macadd'] <= FINAL_STATS['val_macadd']
print('-> dùng TTA:', USE_TTA)

Loading config /usr/local/lib/python3.12/dist-packages/detectron2/model_zoo/configs/COCO-Detection/../Base-RetinaNet.yaml with yaml.unsafe_load. Your machine may be at risk if the file contains malicious content.


[07/06 19:42:23 d2.checkpoint.detection_checkpoint]: [DetectionCheckpointer] Loading from /kaggle/working/r0/model_final.pth ...
no-TTA : {'poison_firing_rate': 0.0184, 'streak_retention': 0.2013, 'val_macadd': 1.2052}
TTA    : {'poison_firing_rate': 0.0061, 'streak_retention': 0.1258, 'val_macadd': 1.2505}
-> dùng TTA: False


## 12. Submission = output trực tiếp model đã de-poison

In [14]:
sample = pd.read_csv(SAMPLE_SUB, dtype={'image_id': str})
def fmt(b, s):
    parts = []
    for (x1,y1,x2,y2), sc in zip(b, s):
        x1,y1 = float(np.clip(x1,0,IMG_W)), float(np.clip(y1,0,IMG_H)); x2,y2 = float(np.clip(x2,0,IMG_W)), float(np.clip(y2,0,IMG_H))
        w,h = x2-x1, y2-y1
        if w>0 and h>0 and sc>0: parts += [f'{float(sc):.6f}', f'{x1:.2f}', f'{y1:.2f}', f'{w:.2f}', f'{h:.2f}']
    return ' '.join(parts) if parts else ' '
pred = {}
for p in tqdm(test_files, desc='infer test'):
    img = load_image(p)
    b,s = predict_tta(final_pr, img) if USE_TTA else predict_image(final_pr, img)
    keep = s > SUBMIT_THRESH; pred[p.stem] = (b[keep], s[keep])
sub = sample.copy()
sub['prediction_string'] = sub['image_id'].map(lambda i: fmt(*pred.get(str(i), E)))
sub['prediction_string'] = sub['prediction_string'].fillna(' ')
sub.loc[sub['prediction_string'].str.strip()=='', 'prediction_string'] = ' '
sub.to_csv(SUBMISSION_PATH, index=False)
total = sum(len(s) for _,s in pred.values()); npred = int((sub['prediction_string'].str.strip()!='').sum())
print(f'saved {SUBMISSION_PATH} | rows={len(sub)} | dets={total} ({total/len(test_files):.3f}/img) | imgs w/ preds: {npred}')
print('so sánh: poisoned ~4.1/img | clean mục tiêu ~1.5/img | empty=0. Bản over-unlearn cũ = 0.068/img.')

infer test: 100%|██████████| 2000/2000 [04:08<00:00,  8.05it/s]

saved /kaggle/working/submission.csv | rows=2000 | dets=139 (0.070/img) | imgs w/ preds: 132
so sánh: poisoned ~4.1/img | clean mục tiêu ~1.5/img | empty=0. Bản over-unlearn cũ = 0.068/img.


## 13. Tổng kết + quy tắc nộp

**CHỈ nộp nếu:** `DE-POISONED val_macadd < POISONED baseline` VÀ `retention` cao (streak được giữ) VÀ `dets/img` gần ~1.5, KHÔNG phải ~0.07.
Nếu không đạt → de-poison chưa thắng, tune tiếp (retain_conf, iters, ewc). KHÔNG nộp bản collapse.

Neo LB: submit bản đạt tiêu chí + empty + poisoned-passthrough để hiệu chuẩn.

In [15]:
print('='*60)
final_report = tta_stats if USE_TTA else FINAL_STATS
print('POISONED   :', {k:round(float(base_stats[k]),4) for k in base_stats})
print('DE-POISONED:', {k:round(float(final_report[k]),4) for k in final_report})
ok = (final_report['val_macadd'] < base_stats['val_macadd']) and (final_report['streak_retention'] >= 0.6)
print('\nĐẠT TIÊU CHÍ NỘP:', ok)
if not ok:
    print(' -> CHƯA nên nộp. val_macadd chưa < baseline hoặc retention < 0.6.')
    print('    Tune: tăng retain_conf nếu ôm nhiễu; giảm iters/tăng ewc nếu vẫn over-unlearn;')
    print('          giảm prune_frac nếu mất streak; xem lại số pseudo-GT/ảnh ở cell 7.')
else:
    print(' -> Model giữ streak + bớt bắn poison + gần clean hơn baseline. Nộp được.')

POISONED   : {'poison_firing_rate': 0.9693, 'streak_retention': 1.0, 'val_macadd': 1.4549}
DE-POISONED: {'poison_firing_rate': 0.0184, 'streak_retention': 0.2013, 'val_macadd': 1.2052}

ĐẠT TIÊU CHÍ NỘP: False
 -> CHƯA nên nộp. val_macadd chưa < baseline hoặc retention < 0.6.
    Tune: tăng retain_conf nếu ôm nhiễu; giảm iters/tăng ewc nếu vẫn over-unlearn;
          giảm prune_frac nếu mất streak; xem lại số pseudo-GT/ảnh ở cell 7.
